In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from PIL import Image, ImageEnhance
from matplotlib.patches import FancyArrowPatch
from shapely.geometry import LineString, MultiLineString, box

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica"]


sites = { "CW":  (6.522984, 3.310817), "EXP": (6.529435, 3.316108),
    "PET": (6.540710, 3.319277),  "MBD": (6.530166, 3.316193),   "AJB": (6.544360, 3.318868), "AJA": (6.537347, 3.323543),}

airport = {"label": "Airport", "lat": 6.5774, "long": 3.3212}
isolo = {"label": "Isolo", "lat": 6.5267484884689635, "long": 3.3182259927113438}

lab_offsets = {"CW": (-300, -80), "EXP": (-280, 180), "PET": (140, 220),
    "MBD": (220, -140), "AJB": (140, 280), "AJA": (220, 90),
    "Airport": (300, 140), "Isolo": (120, 120),}

cols = [ "#182b55", "#5f4e94", "#a291c7", "#900c3f", "#82cbec",  "#d94f21", "#febd2b", "#9aab4b", "#8C92AC", "#9b2226", "#6a4c93", "#1982c4", "#8ac926", "#ff595e", "#FFFDD0",  "#ffca3a", "#8338ec", "#3a86ff",]

site_df = pd.DataFrame([{"label": k, "lat": lat, "long": lon} for k, (lat, lon) in sites.items()])
full_df = pd.concat([site_df, pd.DataFrame([airport]), pd.DataFrame([isolo])], ignore_index=True)
gdf = gpd.GeoDataFrame(full_df, geometry=gpd.points_from_xy(full_df["long"], full_df["lat"]), crs="EPSG:4326").to_crs(epsg=3857)
sample_gdf = gdf[~gdf["label"].isin(["Airport", "Isolo"])].copy()
airport_gdf = gdf[gdf["label"] == "Airport"].copy()
isolo_gdf = gdf[gdf["label"] == "Isolo"].copy()

catchments = gpd.read_file("es_world_delineated_catchments_2026-05-22.json")
if catchments.crs is None:
    catchments = catchments.set_crs("EPSG:4326")
catchments = catchments.to_crs(epsg=3857)
catchments["geometry"] = catchments.geometry.buffer(0)  

catch = catchments.dissolve(by="site", as_index=False)
catch["geometry"] = catch.geometry.buffer(0)

site_names = catch["site"].dropna().astype(str).tolist()
col_map = {site: cols[i % len(cols)] for i, site in enumerate(site_names)}

catch["plot_col"] = catch["site"].map(col_map)
catch["area_m2"] = catch.geometry.area
catch = catch.sort_values("area_m2", ascending=False) 

# print("Catchment colour mapping:")
# for site, c in col_map.items():
#     print(site, c)


drainage = gpd.read_file("es_world_drainage_lines_2026-05-22.json")
if drainage.crs is None:
    drainage = drainage.set_crs("EPSG:4326")
drainage = drainage.to_crs(epsg=3857)

health = gpd.read_file("GRID3_NGA_health_facilities_v2_0_-5661871903075391498.geojson")
health = health[health["facility_level_option"].isin(["General Hospital", "Teaching/Tertiary Hospital", "Specialized Hospital"])].copy()
if health.crs is None:
    health = health.set_crs("EPSG:4326")
health = health.to_crs(epsg=3857)


xmin1, ymin1, xmax1, ymax1 = sample_gdf.total_bounds
xmin2, ymin2, xmax2, ymax2 = catch.total_bounds
xmin3, ymin3, xmax3, ymax3 = isolo_gdf.total_bounds
xmin, ymin = min(xmin1, xmin2, xmin3), min(ymin1, ymin2, ymin3)

xmax, ymax = max(xmax1, xmax2, xmax3), max(ymax1, ymax2, ymax3)

xpad = ypad = 1400
map_xlim = (xmin - xpad, xmax + xpad)
map_ylim = (ymin - ypad, ymax + ypad)

bbox = box(map_xlim[0], map_ylim[0], map_xlim[1], map_ylim[1])
health_clip = health[health.geometry.intersects(bbox)].copy()
drainage_clip = drainage[drainage.geometry.intersects(bbox)].copy()


def add_line_arrows(ax, line_gdf, spacing=1200, arrow_size=8, color="black", alpha=0.8, zorder=3.1):
    for geom in line_gdf.geometry:
        if geom is None or geom.is_empty:
            continue

        if isinstance(geom, LineString):
            lines = [geom]
        elif isinstance(geom, MultiLineString):
            lines = list(geom.geoms)
        else:
            continue

        for line in lines:
            if line.length < spacing:
                continue
            for d in range(int(spacing), int(line.length), int(spacing)):
                p1 = line.interpolate(max(d - 100, 0))
                p2 = line.interpolate(min(d + 100, line.length))
                arrow = FancyArrowPatch((p1.x, p1.y), (p2.x, p2.y), arrowstyle="-|>", mutation_scale=arrow_size, linewidth=0.7, color=color, alpha=alpha, zorder=zorder)
                ax.add_patch(arrow)


fig, ax = plt.subplots(figsize=(6, 6), dpi=1000)
ax.set_xlim(map_xlim)
ax.set_ylim(map_ylim)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.VoyagerNoLabels, zoom=15, interpolation="nearest")

for im in ax.get_images():
    pil = Image.fromarray(im.get_array())
    pil = ImageEnhance.Brightness(pil).enhance(0.78)
    pil = ImageEnhance.Contrast(pil).enhance(1.55)
    pil = ImageEnhance.Sharpness(pil).enhance(1.8)
    im.set_data(pil)

for _, row in catch.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    poly = gpd.GeoSeries([row.geometry], crs=catch.crs)
    poly.plot(ax=ax, facecolor=row["plot_col"], edgecolor=row["plot_col"], alpha=0.14, linewidth=1.2, zorder=2)
    poly.boundary.plot(ax=ax, edgecolor=row["plot_col"], linewidth=1.5, alpha=0.95, zorder=2.3)

drainage_clip.plot(ax=ax, color="black", linewidth=0.45, alpha=0.65, zorder=2.7)
add_line_arrows(ax, drainage_clip, spacing=1200, arrow_size=8, color="black", alpha=0.8, zorder=3.0)

health_clip.plot(ax=ax, marker="o", color="#ff595e", edgecolor="black", linewidth=0.3, markersize=14, alpha=0.9, zorder=3.3, label="Hospital")
isolo_gdf.plot(ax=ax, marker="o", color="#ff595e", edgecolor="black", linewidth=0.3, markersize=14, alpha=0.95, zorder=3.5, label="")

sample_gdf.plot(ax=ax, color="#82cbec", edgecolor="black", linewidth=0.6, markersize=45, zorder=4, label="Sampling site")
airport_gdf.plot(ax=ax, color="#182b55", edgecolor="black", linewidth=0.6, markersize=45, zorder=4, label="Airport")

for _, row in gdf.iterrows():
    if row["label"] == "Isolo":
        continue
    dx, dy = lab_offsets[row["label"]]
    ax.text(row.geometry.x + dx, row.geometry.y + dy, row["label"], fontsize=8, fontweight="bold", color="black", zorder=5)



bar_len = 5000
x0, x1 = ax.get_xlim()
y0, y1 = ax.get_ylim()
bar_x = x0 + 0.08 * (x1 - x0)
bar_y = y0 + 0.06 * (y1 - y0)

ax.plot([bar_x, bar_x + bar_len], [bar_y, bar_y], color="black", linewidth=1.5, zorder=5)
ax.text(bar_x + bar_len / 2, bar_y + 300, "5 km", ha="center", va="bottom", fontsize=8, fontweight="bold", color="black", zorder=5)

ax.legend(frameon=True, fontsize=7, loc="upper right", facecolor="white", edgecolor="black")

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.5)

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()